In [1]:
import time, os
from datetime import date, timedelta
import pandas as pd
from nba_api.stats.endpoints import scoreboardv3, boxscoretraditionalv3

# ---------- Motor ----------
def _clip01(x):
    return max(0.0, min(1.0, x))

def closeness_score(final_margin):
    return _clip01(1 - final_margin / 20.0)

def overtime_score(num_ot):
    if num_ot <= 0:
        return 0.0
    return _clip01(0.7 + 0.3 * (num_ot - 1))

def pace_score(total_points):
    return _clip01((total_points - 200) / 60.0)

def star_score(best_game_score):
    return _clip01((best_game_score - 15) / 30.0)

WEIGHTS = {"closeness": 0.45, "overtime": 0.20, "pace": 0.05, "star": 0.30}

def game_score_col(jug):
    return (jug["points"]
        + 0.4*jug["fieldGoalsMade"] - 0.7*jug["fieldGoalsAttempted"]
        - 0.4*(jug["freeThrowsAttempted"] - jug["freeThrowsMade"])
        + 0.7*jug["reboundsOffensive"] + 0.3*jug["reboundsDefensive"]
        + jug["steals"] + 0.7*jug["assists"] + 0.7*jug["blocks"]
        - 0.4*jug["foulsPersonal"] - jug["turnovers"])

# ---------- Procesar una fecha ----------
def procesar_fecha(fecha):
    sb = scoreboardv3.ScoreboardV3(game_date=fecha, timeout=120)
    dfs = sb.get_data_frames()
    line = dfs[2]
    resumen = line.groupby("gameId")["score"].agg(total="sum", maximo="max", minimo="min")
    resumen["margen"] = resumen["maximo"] - resumen["minimo"]
    if len(resumen) == 0:
        return None
    header = dfs[1].copy()
    header["prorrogas"] = header["period"] - 4
    resumen = resumen.join(header.set_index("gameId")["prorrogas"])
    estrellas, estrella_nombre = {}, {}
    for gid in resumen.index:
        bx = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=gid, timeout=120)
        jug = bx.get_data_frames()[0]
        jug["gs"] = game_score_col(jug)
        mejor = jug.loc[jug["gs"].idxmax()]
        estrellas[gid] = mejor["gs"]
        estrella_nombre[gid] = f'{mejor["firstName"]} {mejor["familyName"]}'
        time.sleep(1)
    enfrentamientos = line.groupby("gameId")["teamTricode"].apply(lambda x: " vs ".join(x))
    resumen["fecha"]       = fecha
    resumen["partido"]     = resumen.index.map(enfrentamientos)
    resumen["estrella_gs"] = resumen.index.map(estrellas)
    resumen["mvp"] = resumen.index.map(estrella_nombre)
    resumen["nota_igualdad"] = resumen["margen"].apply(closeness_score)
    resumen["nota_ritmo"]    = resumen["total"].apply(pace_score)
    resumen["nota_ot"]       = resumen["prorrogas"].apply(overtime_score)
    resumen["nota_estrella"] = resumen["estrella_gs"].apply(star_score)
    resumen["KPI"] = round(100 * (
          WEIGHTS["closeness"]*resumen["nota_igualdad"]
        + WEIGHTS["overtime"] *resumen["nota_ot"]
        + WEIGHTS["pace"]     *resumen["nota_ritmo"]
        + WEIGHTS["star"]     *resumen["nota_estrella"]
    ), 1)
    return resumen.reset_index()

In [2]:
archivo = "kpi_temporada.csv"

# --- Resumible: qué fechas ya están guardadas ---
fechas_hechas = set()
if os.path.exists(archivo):
    ya = pd.read_csv(archivo)
    fechas_hechas = set(ya["fecha"].astype(str).unique())
print("Fechas ya guardadas:", len(fechas_hechas))

# --- Rango a procesar (AHORA: prueba de 3 días) ---
inicio = date(2025, 10, 21)
fin    = date(2026, 6, 30)

dia = inicio
while dia <= fin:
    fstr = dia.strftime("%Y-%m-%d")
    if fstr not in fechas_hechas:
        try:
            res = procesar_fecha(fstr)
            if res is not None and len(res) > 0:
                res.to_csv(archivo, mode="a", header=not os.path.exists(archivo), index=False)
                print(fstr, "->", len(res), "partidos guardados")
            else:
                print(fstr, "-> sin partidos")
        except Exception as e:
            print(fstr, "ERROR:", e)
    dia += timedelta(days=1)

print("\nTerminado.")

Fechas ya guardadas: 0
2025-10-21 -> 2 partidos guardados
2025-10-22 -> 12 partidos guardados
2025-10-23 -> 2 partidos guardados
2025-10-24 -> 12 partidos guardados
2025-10-25 -> 5 partidos guardados
2025-10-26 -> 9 partidos guardados
2025-10-27 -> 11 partidos guardados
2025-10-28 -> 5 partidos guardados
2025-10-29 -> 10 partidos guardados
2025-10-30 -> 4 partidos guardados
2025-10-31 -> 8 partidos guardados
2025-11-01 -> 6 partidos guardados
2025-11-02 -> 8 partidos guardados
2025-11-03 -> 9 partidos guardados
2025-11-04 -> 6 partidos guardados
2025-11-05 -> 11 partidos guardados
2025-11-06 -> 1 partidos guardados
2025-11-07 -> 11 partidos guardados
2025-11-08 -> 8 partidos guardados
2025-11-09 -> 7 partidos guardados
2025-11-10 -> 9 partidos guardados
2025-11-11 -> 6 partidos guardados
2025-11-12 -> 12 partidos guardados
2025-11-13 -> 3 partidos guardados
2025-11-14 -> 9 partidos guardados
2025-11-15 -> 5 partidos guardados
2025-11-16 -> 8 partidos guardados
2025-11-17 -> 8 partidos 

In [3]:
df = pd.read_csv("kpi_temporada.csv")
print("Partidos totales:", len(df))
print("Desde:", df["fecha"].min(), "hasta:", df["fecha"].max())
print("Columnas:", list(df.columns))

Partidos totales: 1329
Desde: 2025-10-21 hasta: 2026-06-13
Columnas: ['gameId', 'total', 'maximo', 'minimo', 'margen', 'prorrogas', 'fecha', 'partido', 'estrella_gs', 'mvp', 'nota_igualdad', 'nota_ritmo', 'nota_ot', 'nota_estrella', 'KPI']
